# Directional (S) and magnitude (M) outcomes — normalised-sum injection

We decompose the joint outcome into two conceptually distinct quantities, following the report.

**Per-behaviour effects** (relative to baseline): the individual effect $\Delta^{\text{ind}}_i = E^{(1,0)}_i - E^{(0,0)}_i$ and the joint effect $\Delta^{\text{joint}}_i = E^{(1,1)}_i - E^{(0,0)}_i$ (the `delta_*_single` and `delta_*_joint` columns).

**Direction — do the behaviours reinforce or suppress one another?**

$$S_i = \Delta^{\text{joint}}_i - \Delta^{\text{ind}}_i, \qquad S(i,j) = \tfrac{1}{2}\big(S_i + S_j\big),$$

positive = reinforced in company, negative = suppressed. We also track the more-suppressed behaviour $\min(S_i, S_j)$.

**Magnitude — how strongly do both behaviours come out together?**

$$M(i,j) = \tfrac{1}{2}\big(\lvert\Delta^{\text{joint}}_i\rvert + \lvert\Delta^{\text{joint}}_j\rvert\big).$$

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parents[1]

In [ ]:
# Load data
with open(REPO_ROOT / "results/composition/v2_phase125_normTrue_a4.5/scoring/summary.json") as f:
    summary = json.load(f)

rows = []
for p in summary["pairs"]:
    if p.get("status") != "ok":
        continue
    rows.append({
        "trait_a": p["trait_a"], "trait_b": p["trait_b"],
        "cos": p["cos"], "regime": p["regime"],
        "delta_a_joint":  p["delta"]["trait_a_joint"],
        "delta_a_single": p["delta"]["trait_a_single"],
        "delta_b_joint":  p["delta"]["trait_b_joint"],
        "delta_b_single": p["delta"]["trait_b_single"],
    })
df = pd.DataFrame(rows)
print(f"n = {len(df)} pairs")
df.head()

In [ ]:
# Directional change per behaviour:  S_i = Δ_joint_i − Δ_ind_i   (Eq. 8)
df["S_a"] = df["delta_a_joint"] - df["delta_a_single"]
df["S_b"] = df["delta_b_joint"] - df["delta_b_single"]

# Directional outcome:        S = ½(S_a + S_b)                   (Eq. 9)
df["S"]     = 0.5 * (df["S_a"] + df["S_b"])
# More-suppressed behaviour:  min(S_a, S_b)
df["S_min"] = df[["S_a", "S_b"]].min(axis=1)
# Joint-expression magnitude: M = ½(|Δ_joint_a| + |Δ_joint_b|)   (Eq. 7)
df["M"]     = 0.5 * (df["delta_a_joint"].abs() + df["delta_b_joint"].abs())

print(df[["S", "S_min", "M"]].describe().round(3))
df[["trait_a", "trait_b", "cos", "regime", "S", "S_min", "M"]].head(10)